# 04 - Modelling, Evaluation, Calibration

**Purpose:** Baseline-first model comparison (Dummy, LogReg, RF, XGBoost, LightGBM), stratified CV, untouched holdout, calibration, thresholds, error and decile analysis.

**Inputs:** data/processed/features_train.parquet, configs/config.yaml

**Outputs / persisted artifacts:** artifacts/models/creditiq_bundle.joblib, artifacts/metrics/*.csv|json, figures/{roc,pr,calibration,confusion}.png

**Dependencies:** Notebook 02

**Execution order:** 4 of 6

**Validation checks:** Holdout never used for fitting/calibration/threshold; metrics.json holds only executed values

_All logic lives in `src/creditiq`; cells are safe to re-run (completed stages are skipped, artifacts live on Google Drive)._

In [ ]:
# --- Colab setup (safe to re-run) ---
import os, sys, subprocess
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
ROOT = Path("/content/drive/MyDrive/CreditIQ") if Path("/content/drive/MyDrive").exists() else Path.cwd().parent
os.environ["CREDITIQ_ROOT"] = str(ROOT); sys.path.insert(0, str(ROOT / "src"))
if not (ROOT / "src").exists():
    raise SystemExit(f"Project not found at {ROOT}. Unzip CreditIQ into MyDrive/CreditIQ first.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=False)
from creditiq.config import get_paths, load_config, read_state
paths = get_paths(); cfg = load_config(paths)
print("Project root:", ROOT); print("Last completed stage:", read_state(paths).get("last_completed"))

In [ ]:
# Edit configs/config.yaml first if needed (cv.sample_n bounds CV runtime; tuning.optuna_trials=0 means no tuning).
from creditiq import pipeline
pipeline.run("train")      # skip-able: set force=True to retrain, CV results are cached

In [ ]:
import pandas as pd, json
display(pd.read_csv(paths.metrics / "cv_comparison.csv")); display(pd.read_csv(paths.metrics / "holdout_comparison.csv"))
m = json.load(open(paths.metrics / "metrics.json")); print(json.dumps(m["test_calibrated"], indent=2)); print(m["calibration_comparison"])
imb = paths.metrics / "imbalance_experiment.csv"
if imb.exists(): display(pd.read_csv(imb))

In [ ]:
from IPython.display import Image, display
for f in ["roc", "pr", "calibration", "confusion"]: display(Image(str(paths.figures / f"{f}.png")))
display(pd.read_csv(paths.metrics / "threshold_analysis.csv")); display(pd.read_csv(paths.metrics / "risk_deciles_test.csv")); display(pd.read_csv(paths.metrics / "error_analysis.csv"))